# 🚀 실전 DPO (Direct Preference Optimization) 얼라인먼트 튜닝

In [1]:
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import DPOTrainer
from datasets import Dataset

## 1. 선호도 데이터셋 구축 (Chosen vs Rejected)
DPO는 [프롬프트, 인간이 좋아하는 답변, 인간이 싫어하는 답변] 세 쌍의 데이터가 필요합니다. 우리는 로봇이 공손한 대답을 하도록 학습시킬 것입니다.

In [2]:
data = {
    "prompt": [
        "너의 이름은 뭐야?",
        "파이썬이 뭐야?",
        "안녕!"
    ],
    "chosen": [
        "제 이름은 친절한 AI 어시스턴트입니다. 무엇을 도와드릴까요?",
        "파이썬은 배우기 쉽고 강력한 프로그래밍 언어입니다.",
        "안녕하세요! 만나서 반갑습니다."
    ],
    "rejected": [
        "내가 왜 알려줘야 해?",
        "구글에 검색해봐.",
        "뭐."
    ]
}

dataset = Dataset.from_dict(data)
print("✅ 선호도 데이터셋 준비 완료:")
print(dataset[0])

✅ 선호도 데이터셋 준비 완료:
{'prompt': '너의 이름은 뭐야?', 'chosen': '제 이름은 친절한 AI 어시스턴트입니다. 무엇을 도와드릴까요?', 'rejected': '내가 왜 알려줘야 해?'}


## 2. 모델 로드 및 DPOTrainer 실행
HuggingFace `trl` 라이브러리의 `DPOTrainer`는 내부적으로 $\pi_\theta$와 $\pi_{ref}$의 로그 확률(Log-probs) 차이를 계산하여 DPO Loss를 역전파합니다.

In [3]:
# 로컬 환경을 위해 극도로 작은 모델 사용 (gpt2)
model_name = "sshleifer/tiny-gpt2" 
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# DPO 하이퍼파라미터 설정
training_args = TrainingArguments(
    output_dir="./dpo_output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    learning_rate=1e-5,
    remove_unused_columns=False,
    report_to="none" # wandb 비활성화
)

# DPOTrainer 인스턴스화
# 참고: 원래 DPO는 ref_model이 필요하지만, None으로 두면 내부에서 현재 모델을 깊은 복사하여 사용합니다.
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None, 
    args=training_args,
    beta=0.1, # DPO 식의 beta: Reference 모델에서 얼마나 벗어날지 허용하는 페널티 계수
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print("🚀 DPO 훈련 시작... (Logits가 Chosen 쪽으로 서서히 이동합니다)")
dpo_trainer.train()
print("✅ DPO 얼라인먼트 튜닝 완료! 모델이 더욱 '공손해지는' 방향으로 가중치가 조정되었습니다.")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

TypeError: DPOTrainer.__init__() got an unexpected keyword argument 'beta'